# Inland Excess Water (IEW) Monitoring — Soil-Severity Specialist

**Notebook 15. A second-stage model that only decides *how badly* a soil pixel is affected.**

### Why this notebook

Every model so far (notebooks 10 to 14) reaches about 98 F1 on water and not-affected, 75 on vegetation at best, and **60 to 70 F1 on the three soil-severity classes** (seriously / moderately / slightly affected). Notebook 12, the best model, still scores 69.6 / 62.3 / 66.7 on them. The three soil grades are neighbouring points on one scale, they cover under 3 % of the pixels, and a network that must also separate water, vegetation and background spends little capacity on telling the grades apart.

This notebook splits the job in two:

```
 Stage A  (notebook 12's trained PT-UPerNet, frozen)
     inputs ──► 6-class probabilities  pA          "is it soil?"  (soil mass = pA[serious] + pA[moderate] + pA[slight])
            └─► 64-channel feature map F (last layer before its output heads)

 Stage B  (this notebook: the Soil-Severity Specialist, trained on soil pixels only)
     [ F  +  terrain, spectral indices, radar, validity flags ] ──► dilated-convolution context network
            ──► ordinal head (P(>= moderate), P(serious | >= moderate)) + 3-way grade head
            ──► grade probabilities pS over (serious, moderate, slight)

 Fusion:   inside the soil mass, mix the grade distributions
           soil = pA[1] + pA[2] + pA[3]
           grade = (1 - beta) * pA[1:4] / soil  +  beta * pS
           final probabilities = pA with pA[1:4] replaced by soil * grade
```

- **Trained only where it matters.** The loss is computed on pixels whose ground truth is soil, with an ordinal structure (slight < moderate < serious), label smoothing toward the neighbouring grade, and class-balanced weights. Patches are sampled so that every epoch is rich in soil.
- **Large context.** A stack of dilated convolutions at ¼ resolution plus a global-pooling term gives the specialist a view of several kilometres, so landscape position (from the terrain channels) can inform the grade.
- **Cheap to train.** Stage A is frozen, so only a small network trains. Minutes per epoch, no GPU-memory problems.
- **A safety valve.** `beta` is chosen on the two held-out training scenes. If the specialist does not improve held-out macro-F1 by at least 0.2 points, `beta = 0` and the output is exactly notebook 12's.

### What you get besides the final score

- A **grading-only comparison** on the test set: among pixels that are truly soil, how well do notebook 12's own grade choice and the specialist's choice match the ground truth? This separates "can the specialist grade better?" from "does the detection of soil limit the final score?".
- An inference-time ablation of the terrain inputs.

### Honest expectations

- The gain is **bounded by soil detection**: the specialist only re-grades pixels that stage A already believes are soil.
- **Stacking caveat.** Stage A's features on the *training* scenes are cleaner than on unseen scenes, because stage A was fitted to them. The specialist therefore trains on slightly optimistic features. The held-out scenes (never seen by stage A) are used for every selection step (best epoch, `beta`), and feature noise and channel dropout are applied during training, but the effect cannot be removed completely.
- The serious/moderate/slight boundaries in the ground truth are themselves fuzzy, so a large jump is unlikely. **90+ F1 on every class remains unlikely.**

### How to run

1. **Prerequisite:** notebook 12 must have been run to completion, which creates `outputs/checkpoints/ptuper_best.pt`. Checkpoints are git-ignored, so this file exists only on the machine that ran notebook 12; copy it into `outputs/checkpoints/` on the machine you use now.
2. **Environment:** the same as notebook 12 (`torchgeo`, `rasterio`, `pyyaml`, `matplotlib`, a CUDA PyTorch). The ResNet architecture is built without downloading weights because the checkpoint supplies them.
3. **Terrain:** the files in `outputs/dem/*.npy` from notebook 12 are reused; otherwise the Copernicus tiles are downloaded again.
4. **Dry run:** `QUICK_TEST = True`, run all cells; then set it to `False`, **restart the kernel** and run all cells.
5. **Memory:** with stage A frozen, `BATCH_SIZE = 8` at 256 px needs roughly 3 to 4 GB. Reduce to 4 if needed.
6. Training resumes automatically if interrupted.

This code has **not been executed** at the time of writing; expect the dry run to show small fixes.

### Outputs

- `outputs/checkpoints/soil_best.pt`, `soil_last.pt` (the specialist only, a few MB), `soil_calibration.json` (`beta`)
- `outputs/soil_results.json` (all metrics), `outputs/preds_soil/<scene>.npy` (label maps: stage A, final)

## 1. Imports and configuration

In [ ]:
import json
import math
import time
import urllib.request
from pathlib import Path

import numpy as np
import yaml
import rasterio
from rasterio.windows import Window
from rasterio.warp import reproject, Resampling, transform_bounds
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

QUICK_TEST = False          # True = tiny dry run of the whole notebook (writes *_quicktest* files only)

SEED         = 0
PATCH        = 256
GRID_STRIDE  = 64
BATCH_SIZE   = 8            # stage A is frozen, so this is cheap
NUM_WORKERS  = 0            # keep 0 on Windows + Jupyter
USE_AMP      = True
RESUME       = True

PRETRAINED   = False        # stage A's weights come from its checkpoint, nothing to download
USE_DEM      = True         # terrain features; reuses outputs/dem/*.npy from notebook 12
STAGE_A_CKPT = Path("outputs/checkpoints/ptuper_best.pt")      # written by notebook 12

EPOCHS            = 30
SAMPLES_PER_EPOCH = 2000
VAL_PATCHES       = 300
SPEC_LR           = 1e-3
WEIGHT_DECAY      = 0.01
WARMUP_FRAC       = 0.1
FEAT_NOISE        = 0.10    # training: Gaussian noise on stage A's features (relative to their std)
ORDINAL_SMOOTH    = 0.10
GAIN_JITTER       = 0.05

TILE, MARGIN = 512, 64      # same inference windows as notebook 12
TTA          = "flip"       # "none" | "flip" | "d4"
MIN_GAIN     = 0.002        # beta stays 0 unless held-out macro-F1 improves by at least this much (0.2 points)

VAL_SCENES  = ["BNPI_20230208", "KNPI_20220310"]       # same as notebooks 10 to 14
# Soil-rich sampling: this notebook trains only on soil pixels (classes 1, 2, 3 = serious, moderate, slight)
CLASS_SHARE = {1: 0.33, 2: 0.33, 3: 0.29}

MAX_TILES = None
if QUICK_TEST:
    EPOCHS, SAMPLES_PER_EPOCH, VAL_PATCHES = 2, 16, 8
    TTA, MAX_TILES = "none", 2

TAG       = "soil" + ("_quicktest" if QUICK_TEST else "")
CKPT_DIR  = Path("outputs/checkpoints")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
SPEC_BEST = CKPT_DIR / f"{TAG}_best.pt"
SPEC_LAST = CKPT_DIR / f"{TAG}_last.pt"
CAL_PATH  = CKPT_DIR / f"{TAG}_calibration.json"
RESULTS_PATH = Path("outputs") / f"{TAG}_results.json"
PRED_DIR  = Path("outputs") / f"preds_{TAG}"
PRED_DIR.mkdir(parents=True, exist_ok=True)
DEM_DIR = Path("outputs/dem")
DEM_RAW = DEM_DIR / "raw"
DEM_RAW.mkdir(parents=True, exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ON = USE_AMP and device.type == "cuda"
AMP_DTYPE = torch.bfloat16 if (AMP_ON and torch.cuda.is_bf16_supported()) else torch.float16
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    print(f"GPU: {torch.cuda.get_device_name(0)} | "
          f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB")
else:
    print("WARNING: no CUDA GPU found - this notebook is meant for a GPU. Use QUICK_TEST only on CPU.")
print(f"PyTorch {torch.__version__} | device {device} | AMP {AMP_ON} ({AMP_DTYPE if AMP_ON else '-'})")


def autocast():
    return torch.autocast(device_type=device.type, dtype=AMP_DTYPE, enabled=AMP_ON)


def make_scaler():
    enabled = AMP_ON and AMP_DTYPE == torch.float16
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)

## 2. Data: scenes and per-pixel inputs

Identical to notebook 12, so stage A receives exactly the inputs it was trained on: 11 Sentinel-2 bands (reflectance), 2 Sentinel-1 channels (standardised with the dataset statistics), and a 15-channel auxiliary stack (6 spectral indices, cloud score, validity flags, VV/VH ratio, 5 terrain features from the Copernicus DEM).

In [ ]:
with open("configs/data.yaml", "r") as f:
    config = yaml.safe_load(f)

dataset_root = Path(config["dataset_root"])
LOCAL_ROOT = Path("C:/E/S5/Deep_Learning/dataset/tifs_FINAL")
if not dataset_root.exists() and LOCAL_ROOT.exists():
    dataset_root = LOCAL_ROOT
assert dataset_root.exists(), f"Dataset not found at {dataset_root} - edit dataset_root in configs/data.yaml"
print(f"Dataset root: {dataset_root}")

IGNORE = 255
NUM_CLASSES = 6
CLASS_NAMES = [
    "Open surface water (raw 2)",
    "Soil seriously affected (raw 3)",
    "Soil moderately affected (raw 4)",
    "Soil slightly affected (raw 5)",
    "Vegetation affected (raw 6)",
    "Not affected (raw 7)",
]
GT_LUT = np.full(256, IGNORE, dtype=np.uint8)
for raw, cls in {2: 0, 3: 1, 4: 2, 5: 3, 6: 4, 7: 5}.items():
    GT_LUT[raw] = cls

# Band order: B01 B02 B03 B04 B05 B06 B07 B08 B8A B11 B12
S2_MEAN = np.array([0.0436, 0.0557, 0.0723, 0.0838, 0.1151, 0.1598,
                    0.1800, 0.1974, 0.2004, 0.2233, 0.1706], dtype=np.float32)
S2_STD  = np.array([0.0182, 0.0233, 0.0272, 0.0340, 0.0380, 0.0598,
                    0.0686, 0.0751, 0.0728, 0.0674, 0.0606], dtype=np.float32)
S1_MEAN = np.array([-21.6081, -13.6686, -8.0403], dtype=np.float32)
S1_STD  = np.array([3.4706, 3.0644, 2.5309], dtype=np.float32)

N_DEM   = 5
AUX_CH  = 6 + 1 + 1 + 1 + 1 + N_DEM     # indices, cloud, S2 valid, VV/VH ratio, S1 valid, terrain
FULL_CH = 11 + 2 + AUX_CH               # inputs of the full-resolution refinement branch


def make_inputs(s2_raw: np.ndarray, s1_raw: np.ndarray, dem):
    """Raw window arrays -> s2 (11,h,w) reflectance, s1 (2,h,w) standardised, aux (15,h,w)."""
    s2 = s2_raw.astype(np.float32)
    v2 = np.isfinite(s2).all(0)
    s2 = np.nan_to_num(s2, nan=0.0, posinf=0.0, neginf=0.0) / 10000.0
    v2 &= s2.sum(0) > 0
    s2 = np.clip(s2, 0.0, 2.0) * v2[None]

    eps = 1e-6
    blue, green, red, nir, nir_n, swir1, swir2 = s2[1], s2[2], s2[3], s2[7], s2[8], s2[9], s2[10]
    indices = np.stack([
        (nir - red) / (nir + red + eps),            # NDVI
        (nir_n - swir1) / (nir_n + swir1 + eps),    # NDMI
        (green - nir) / (green + nir + eps),        # NDWI
        (green - swir1) / (green + swir1 + eps),    # MNDWI
        (swir1 - red) / (swir1 + red + eps),        # SWIR1-RED soil-moisture index
        (swir1 - swir2) / (swir1 + swir2 + eps),    # SWIR ratio
    ]) * 2.0
    cloud = np.clip((np.minimum(blue, red) - 0.10) / 0.20, 0.0, 1.0)

    s1 = s1_raw.astype(np.float32)
    v1 = np.isfinite(s1).all(0)
    s1 = np.nan_to_num(s1, nan=0.0, posinf=0.0, neginf=0.0)
    v1 &= np.abs(s1[0]) > 1e-6
    s1n = np.clip((s1 - S1_MEAN[:, None, None]) / S1_STD[:, None, None], -6.0, 6.0) * v1[None]

    h, w = s2.shape[1:]
    terrain = dem.astype(np.float32) if dem is not None else np.zeros((N_DEM, h, w), dtype=np.float32)
    f2, f1 = v2.astype(np.float32), v1.astype(np.float32)
    aux = np.concatenate([indices * f2[None], (cloud * f2)[None], f2[None], s1n[2:3], f1[None], terrain], axis=0)
    aux = np.clip(np.nan_to_num(aux, nan=0.0, posinf=6.0, neginf=-6.0), -6.0, 6.0)
    return s2.astype(np.float32), s1n[:2].astype(np.float32), aux.astype(np.float32)


class Scene:
    """One acquisition. S2/S1 read by window; GT and terrain held in RAM."""

    def __init__(self, split: str, split_dir: Path, filename: str):
        self.name = Path(filename).stem
        self.split = split
        self.site = self.name.split("_")[0]
        self.s2_path = split_dir / "S2" / filename
        self.s1_path = split_dir / "S1" / filename
        self.gt_path = split_dir / "GT" / filename
        for p in (self.s1_path, self.gt_path):
            if not p.exists():
                raise FileNotFoundError(f"Missing file: {p}")
        with rasterio.open(self.s2_path) as src:
            self.H, self.W = src.height, src.width
            self.transform, self.crs, self.bounds = src.transform, src.crs, src.bounds
        self.dem = None              # (5,H,W) float16, filled in §3
        self._gt = None
        self._s2 = None
        self._s1 = None

    @property
    def gt(self) -> np.ndarray:
        if self._gt is None:
            with rasterio.open(self.gt_path) as src:
                raw = np.nan_to_num(src.read(1), nan=0.0)
            gt = GT_LUT[np.clip(raw, 0, 255).astype(np.uint8)]
            assert gt.shape == (self.H, self.W), f"GT size mismatch in {self.name}"
            self._gt = gt
        return self._gt

    def read(self, r: int, c: int, h: int, w: int):
        if self._s2 is None:
            self._s2 = rasterio.open(self.s2_path)
            self._s1 = rasterio.open(self.s1_path)
        win = Window(c, r, w, h)
        dem = self.dem[:, r:r + h, c:c + w] if self.dem is not None else None
        return self._s2.read(window=win), self._s1.read(window=win), dem

    def __getstate__(self):
        state = self.__dict__.copy()
        state["_s2"] = state["_s1"] = None
        return state


def load_split(split: str):
    split_dir = dataset_root / "tifs" / split
    if not split_dir.exists():
        split_dir = dataset_root / split
    files = sorted(p.name for p in (split_dir / "S2").glob("*.tif"))
    if not files:
        raise FileNotFoundError(f"No .tif files under {split_dir / 'S2'}")
    return [Scene(split, split_dir, fn) for fn in files]


all_train = load_split("Train")
all_test  = load_split("Test")
missing = set(VAL_SCENES) - {s.name for s in all_train}
assert not missing, f"VAL_SCENES not found in Train split: {missing}"
train_scenes = [s for s in all_train if s.name not in VAL_SCENES]
val_scenes   = [s for s in all_train if s.name in VAL_SCENES]
print(f"Train scenes: {len(train_scenes)} | held-out val scenes: {[s.name for s in val_scenes]} | test scenes: {len(all_test)}")
print(f"CRS of first scene: {all_train[0].crs} | pixel size: {abs(all_train[0].transform.a):.1f} m")

## 3. Terrain features from the Copernicus 30 m DEM

1. **Download.** For each site, convert the scene bounds to latitude/longitude and fetch the 1°×1° Copernicus GLO-30 tiles covering it from the public AWS bucket (`copernicus-dem-30m`, no account needed). Tiles already in `outputs/dem/raw/` are reused, so you can also drop tiles there by hand.
2. **Align.** Reproject (bilinear) every overlapping tile onto the scene's exact 10 m pixel grid.
3. **Features (5 channels).** Each is scaled and clipped to ±6:
   - relative elevation: height − site median, divided by 10 m;
   - slope in degrees, divided by 2;
   - **TPI at 150 m, 500 m and 1.5 km**: height − mean height in a square window of that radius, divided by 2 m. Negative TPI means the pixel sits in a depression, where excess water collects.
4. Results are cached once per site grid as `.npy` files. All dates of a site share one grid, so the DEM is computed only once per site.

The DEM is static external data, so it causes no label or test leakage. Expect it to matter mostly for the soil-severity classes. §9 measures it.

In [ ]:
def cop_dem_tile(lat: int, lon: int):
    ns, ew = ("N" if lat >= 0 else "S"), ("E" if lon >= 0 else "W")
    name = f"Copernicus_DSM_COG_10_{ns}{abs(lat):02d}_00_{ew}{abs(lon):03d}_00_DEM"
    return name, f"https://copernicus-dem-30m.s3.amazonaws.com/{name}/{name}.tif"


def ensure_dem_tiles(sc: Scene):
    w, s, e, n = transform_bounds(sc.crs, "EPSG:4326", *sc.bounds, densify_pts=21)
    for lat in range(math.floor(s), math.floor(n) + 1):
        for lon in range(math.floor(w), math.floor(e) + 1):
            name, url = cop_dem_tile(lat, lon)
            path = DEM_RAW / f"{name}.tif"
            if path.exists():
                continue
            tmp = path.with_suffix(".part")
            try:
                print(f"  downloading {name} ...")
                urllib.request.urlretrieve(url, tmp)
                tmp.rename(path)
            except Exception as ex:
                if tmp.exists():
                    tmp.unlink()
                print(f"  WARNING: could not download {name} ({ex}). Put the tile in {DEM_RAW} manually if needed.")


def dem_on_grid(sc: Scene) -> np.ndarray:
    dst = np.full((sc.H, sc.W), np.nan, dtype=np.float32)
    for tif in sorted(DEM_RAW.glob("*.tif")):
        with rasterio.open(tif) as src:
            l, b, r, t = transform_bounds(src.crs, sc.crs, *src.bounds, densify_pts=21)
            if r <= sc.bounds.left or l >= sc.bounds.right or t <= sc.bounds.bottom or b >= sc.bounds.top:
                continue
            tmp = np.full_like(dst, np.nan)
            reproject(source=rasterio.band(src, 1), destination=tmp,
                      src_transform=src.transform, src_crs=src.crs, src_nodata=src.nodata,
                      dst_transform=sc.transform, dst_crs=sc.crs, dst_nodata=np.nan,
                      resampling=Resampling.bilinear)
            fill = np.isnan(dst) & np.isfinite(tmp)
            dst[fill] = tmp[fill]
    return dst


def box_mean(a: np.ndarray, r: int) -> np.ndarray:
    """Mean over a (2r+1)x(2r+1) window, reflect-padded at the borders (integral image)."""
    k = 2 * r + 1
    p = np.pad(a.astype(np.float64), r, mode="reflect")
    I = np.zeros((p.shape[0] + 1, p.shape[1] + 1), dtype=np.float64)
    I[1:, 1:] = p.cumsum(0).cumsum(1)
    return ((I[k:, k:] - I[:-k, k:] - I[k:, :-k] + I[:-k, :-k]) / (k * k)).astype(np.float32)


def dem_features(dem: np.ndarray, pixel_m: float) -> np.ndarray:
    valid = np.isfinite(dem)
    if valid.mean() < 0.5:
        raise RuntimeError(f"DEM covers only {100 * valid.mean():.0f}% of the scene - check tiles in {DEM_RAW}")
    med = float(np.nanmedian(dem))
    dem = np.where(valid, dem, med).astype(np.float32)
    gy, gx = np.gradient(dem, pixel_m)
    slope = np.degrees(np.arctan(np.hypot(gx, gy)))
    radii = [max(1, int(round(m / pixel_m))) for m in (150, 500, 1500)]
    feats = [(dem - med) / 10.0, slope / 2.0] + [(dem - box_mean(dem, r)) / 2.0 for r in radii]
    return np.clip(np.stack(feats), -6.0, 6.0).astype(np.float16)


def grid_key(sc: Scene) -> str:
    return f"{sc.site}_{sc.H}x{sc.W}_{int(round(sc.transform.c))}_{int(round(sc.transform.f))}"


if USE_DEM:
    cache = {}
    for sc in all_train + all_test:
        key = grid_key(sc)
        if key not in cache:
            path = DEM_DIR / f"{key}.npy"
            if path.exists():
                cache[key] = np.load(path)
            else:
                t0 = time.time()
                print(f"Terrain features for {key}:")
                ensure_dem_tiles(sc)
                cache[key] = dem_features(dem_on_grid(sc), abs(sc.transform.a))
                np.save(path, cache[key])
                print(f"  done in {time.time() - t0:.0f}s -> {path}")
        sc.dem = cache[key]
    print(f"Terrain features ready for {len(cache)} site grid(s): {list(cache)}")

    first = all_train[0]
    names = ["relative elevation", "slope", "TPI 150 m", "TPI 500 m", "TPI 1.5 km"]
    fig, axes = plt.subplots(1, N_DEM + 1, figsize=(4 * (N_DEM + 1), 4))
    step = 8
    for k in range(N_DEM):
        im = axes[k].imshow(first.dem[k, ::step, ::step].astype(np.float32), cmap="terrain" if k == 0 else "RdBu",
                            vmin=None if k < 2 else -3, vmax=None if k < 2 else 3)
        axes[k].set_title(names[k]); axes[k].axis("off")
    soil = np.isin(first.gt[::step, ::step], [0, 1, 2, 3]).astype(np.float32)
    axes[-1].imshow(first.dem[4, ::step, ::step].astype(np.float32), cmap="gray", vmin=-3, vmax=3)
    axes[-1].imshow(np.ma.masked_where(soil == 0, soil), cmap="autumn", alpha=0.6)
    axes[-1].set_title(f"{first.name}: water + affected soil over TPI 1.5 km"); axes[-1].axis("off")
    plt.tight_layout(); plt.show()
else:
    print("USE_DEM = False - terrain channels are zero.")

## 3. Patch sampling and dataset

The same class-aware sampler and dataset as notebook 12, but `CLASS_SHARE` now draws 95 % of every epoch's patches from locations that contain serious, moderate or slightly affected soil.

In [ ]:
def window_counts(mask: np.ndarray, size: int, rows: np.ndarray, cols: np.ndarray) -> np.ndarray:
    H, W = mask.shape
    integral = np.zeros((H + 1, W + 1), dtype=np.int32)
    integral[1:, 1:] = mask.cumsum(0, dtype=np.int32).cumsum(1, dtype=np.int32)
    r0, c0 = rows[:, None], cols[None, :]
    return (integral[r0 + size, c0 + size] - integral[r0, c0 + size]
            - integral[r0 + size, c0] + integral[r0, c0])


class PatchIndex:
    def __init__(self, scenes, patch: int, stride: int, min_valid: float = 0.10):
        sid, rr, cc, cnt = [], [], [], []
        for i, sc in enumerate(scenes):
            rows = np.arange(0, sc.H - patch + 1, stride)
            cols = np.arange(0, sc.W - patch + 1, stride)
            counts = np.stack([window_counts(sc.gt == k, patch, rows, cols)
                               for k in range(NUM_CLASSES)], axis=-1).reshape(-1, NUM_CLASSES)
            R, C = np.meshgrid(rows, cols, indexing="ij")
            keep = counts.sum(1) >= min_valid * patch * patch
            sid.append(np.full(int(keep.sum()), i, dtype=np.int32))
            rr.append(R.ravel()[keep]); cc.append(C.ravel()[keep]); cnt.append(counts[keep])
        self.scene_id = np.concatenate(sid)
        self.r = np.concatenate(rr).astype(np.int32)
        self.c = np.concatenate(cc).astype(np.int32)
        self.counts = np.concatenate(cnt).astype(np.int64)

    def __len__(self):
        return len(self.scene_id)

    def sample(self, n: int, rng: np.random.Generator) -> np.ndarray:
        picks = []
        for cls, share in CLASS_SHARE.items():
            w = np.sqrt(self.counts[:, cls].astype(np.float64))
            if w.sum() > 0:
                picks.append(rng.choice(len(self), size=int(round(n * share)), p=w / w.sum()))
        n_rest = n - sum(len(p) for p in picks)
        picks.append(rng.choice(len(self), size=n_rest))
        idx = np.concatenate(picks)
        rng.shuffle(idx)
        return idx


class PatchDataset(Dataset):
    def __init__(self, scenes, index, picks, patch: int, train: bool, jitter: int = 0, seed: int = 0):
        self.scenes, self.index, self.picks = scenes, index, picks
        self.patch, self.train, self.jitter, self.seed = patch, train, jitter, seed

    def __len__(self):
        return len(self.picks)

    def __getitem__(self, i):
        k = self.picks[i]
        sc = self.scenes[self.index.scene_id[k]]
        r, c = int(self.index.r[k]), int(self.index.c[k])
        P = self.patch
        rng = np.random.default_rng((self.seed, int(i))) if self.train else None
        if self.train and self.jitter:
            r = int(np.clip(r + rng.integers(-self.jitter, self.jitter + 1), 0, sc.H - P))
            c = int(np.clip(c + rng.integers(-self.jitter, self.jitter + 1), 0, sc.W - P))
        s2_raw, s1_raw, dem = sc.read(r, c, P, P)
        if self.train and GAIN_JITTER > 0:
            gains = rng.uniform(1 - GAIN_JITTER, 1 + GAIN_JITTER, size=(s2_raw.shape[0], 1, 1)).astype(np.float32)
            s2_raw = s2_raw.astype(np.float32) * gains
        s2, s1, aux = make_inputs(s2_raw, s1_raw, dem)
        y = sc.gt[r:r + P, c:c + P]
        if self.train:
            if rng.random() < 0.5:
                s2, s1, aux, y = s2[..., ::-1], s1[..., ::-1], aux[..., ::-1], y[:, ::-1]
            if rng.random() < 0.5:
                s2, s1, aux, y = s2[..., ::-1, :], s1[..., ::-1, :], aux[..., ::-1, :], y[::-1, :]
            k90 = int(rng.integers(0, 4))
            if k90:
                s2, s1, aux = (np.rot90(a, k90, axes=(-2, -1)) for a in (s2, s1, aux))
                y = np.rot90(y, k90, axes=(0, 1))
        return {
            "s2": torch.from_numpy(np.ascontiguousarray(s2)),
            "s1": torch.from_numpy(np.ascontiguousarray(s1)),
            "aux": torch.from_numpy(np.ascontiguousarray(aux)),
            "y": torch.from_numpy(np.ascontiguousarray(y).astype(np.int64)),
        }


def make_loader(dataset):
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                      pin_memory=device.type == "cuda", drop_last=False)


def to_dev(batch):
    return {k: v.to(device, non_blocking=True) for k, v in batch.items()}


t0 = time.time()
train_index = PatchIndex(train_scenes, PATCH, GRID_STRIDE)
val_index   = PatchIndex(val_scenes, PATCH, PATCH)
print(f"Indexes built in {time.time() - t0:.0f}s | train locations {len(train_index):,} | val locations {len(val_index):,}")

train_pixels = np.zeros(NUM_CLASSES, dtype=np.int64)
for sc in train_scenes:
    train_pixels += np.bincount(sc.gt.ravel(), minlength=256)[:NUM_CLASSES]
freq = train_pixels / train_pixels.sum()
class_weights = (1.0 / freq) ** 0.5
class_weights = torch.tensor(class_weights / class_weights.mean(), dtype=torch.float32)
coarse_pixels = np.array([train_pixels[0], train_pixels[1:4].sum(), train_pixels[4], train_pixels[5]], dtype=np.float64)
coarse_weights = (coarse_pixels.sum() / coarse_pixels) ** 0.5
coarse_weights = torch.tensor(coarse_weights / coarse_weights.mean(), dtype=torch.float32)
n_ser, n_mod, n_sli = train_pixels[1], train_pixels[2], train_pixels[3]
ORD_POS_WEIGHT = torch.tensor([np.clip(n_sli / max(n_mod + n_ser, 1), 0.5, 4.0),
                               np.clip(n_mod / max(n_ser, 1), 0.5, 4.0)], dtype=torch.float32)

print("\nClass frequency in training scenes and loss weights:")
for k in range(NUM_CLASSES):
    print(f"  {CLASS_NAMES[k]:<34s} {100 * freq[k]:6.2f} %   weight {class_weights[k]:.2f}")

val_picks = val_index.sample(VAL_PATCHES, np.random.default_rng(SEED + 12345))
val_loader = make_loader(PatchDataset(val_scenes, val_index, val_picks, PATCH, train=False))

## 4. Stage A: notebook 12's trained model, frozen

The model architecture is rebuilt exactly as in notebook 12 and the best EMA weights are loaded from `outputs/checkpoints/ptuper_best.pt`. A forward hook captures the 64-channel feature map of its last layer before the output heads. The model stays in evaluation mode and no gradients flow into it.

In [ ]:
S2_KEEP = [0, 1, 2, 3, 4, 5, 6, 7, 8, 11, 12]     # SSL4EO 13-band order minus B9 (idx 9) and B10 (idx 10)


def load_ssl4eo_resnet(kind: str):
    """kind = 's2' or 's1' -> timm ResNet-50 (SSL4EO-S12 MoCo weights if PRETRAINED)."""
    try:
        from torchgeo.models import resnet50, ResNet50_Weights
    except ImportError as ex:
        raise ImportError("This notebook needs torchgeo: pip install torchgeo") from ex
    name = "SENTINEL2_ALL_MOCO" if kind == "s2" else "SENTINEL1_ALL_MOCO"
    if PRETRAINED:
        if not hasattr(ResNet50_Weights, name):
            raise AttributeError(f"torchgeo has no ResNet50_Weights.{name}; available: "
                                 f"{[w.name for w in ResNet50_Weights]} - upgrade torchgeo")
        net = resnet50(weights=getattr(ResNet50_Weights, name))
    else:
        net = resnet50(weights=None, in_chans=13 if kind == "s2" else 2)
    if kind == "s2":
        old = net.conv1
        assert old.in_channels == 13, f"expected a 13-band conv1, got {old.in_channels}"
        new = nn.Conv2d(11, old.out_channels, old.kernel_size, old.stride, old.padding, bias=False)
        with torch.no_grad():
            new.weight.copy_(old.weight[:, S2_KEEP] * (13.0 / 11.0))
        net.conv1 = new
    return net


class ResNetEncoder(nn.Module):
    def __init__(self, net):
        super().__init__()
        self.stem = nn.Sequential(net.conv1, net.bn1, net.act1)
        self.pool = net.maxpool
        self.layers = nn.ModuleList([net.layer1, net.layer2, net.layer3, net.layer4])

    def forward(self, x):
        half = self.stem(x)
        y, feats = self.pool(half), []
        for layer in self.layers:
            y = layer(y)
            feats.append(y)
        return half, feats                     # 1/2 (64 ch); 1/4, 1/8, 1/16, 1/32 (256..2048 ch)


def cna(cin: int, cout: int, k: int = 3, s: int = 1) -> nn.Sequential:
    return nn.Sequential(nn.Conv2d(cin, cout, k, s, k // 2, bias=False), nn.GroupNorm(8, cout), nn.GELU())


class AuxEncoder(nn.Module):
    def __init__(self, cin: int, dims=(64, 128, 192, 256)):
        super().__init__()
        self.stem = nn.Sequential(cna(cin, 32), cna(32, dims[0], s=2), cna(dims[0], dims[0], s=2), cna(dims[0], dims[0]))
        self.stages = nn.ModuleList([nn.Sequential(cna(dims[i - 1], dims[i], s=2), cna(dims[i], dims[i]))
                                     for i in range(1, len(dims))])

    def forward(self, x):
        y = self.stem(x)
        feats = [y]
        for st in self.stages:
            y = st(y)
            feats.append(y)
        return feats


class FuseBlock(nn.Module):
    def __init__(self, cin: int, cout: int = 256):
        super().__init__()
        self.proj = nn.Sequential(nn.Conv2d(cin, cout, 1, bias=False), nn.GroupNorm(8, cout), nn.GELU(), cna(cout, cout))
        self.gate = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(cout, cout // 8, 1), nn.GELU(),
                                  nn.Conv2d(cout // 8, cout, 1), nn.Sigmoid())

    def forward(self, *xs):
        y = self.proj(torch.cat(xs, dim=1))
        return y * self.gate(y)


class UPerHead(nn.Module):
    def __init__(self, ch: int = 256, n_levels: int = 4, bins=(1, 2, 3, 6)):
        super().__init__()
        self.ppm = nn.ModuleList([nn.Sequential(nn.AdaptiveAvgPool2d(b), nn.Conv2d(ch, 64, 1, bias=False),
                                                nn.GroupNorm(8, 64), nn.GELU()) for b in bins])
        self.ppm_out = cna(ch + 64 * len(bins), ch)
        self.smooth = nn.ModuleList([cna(ch, ch) for _ in range(n_levels - 1)])
        self.fuse = cna(ch * n_levels, ch)

    def forward(self, feats):
        top = feats[-1]
        pooled = [F.interpolate(m(top), size=top.shape[-2:], mode="bilinear", align_corners=False) for m in self.ppm]
        p = [None] * len(feats)
        p[-1] = self.ppm_out(torch.cat([top] + pooled, dim=1))
        for i in range(len(feats) - 2, -1, -1):
            p[i] = self.smooth[i](feats[i] + F.interpolate(p[i + 1], size=feats[i].shape[-2:],
                                                           mode="bilinear", align_corners=False))
        size = p[0].shape[-2:]
        return self.fuse(torch.cat([p[0]] + [F.interpolate(x, size=size, mode="bilinear", align_corners=False)
                                             for x in p[1:]], dim=1))


class PTUPerNet(nn.Module):
    RES_CH = (256, 512, 1024, 2048)
    AUX_DIMS = (64, 128, 192, 256)

    def __init__(self, num_classes: int = NUM_CLASSES, ch: int = 256):
        super().__init__()
        self.enc_s2 = ResNetEncoder(load_ssl4eo_resnet("s2"))
        self.enc_s1 = ResNetEncoder(load_ssl4eo_resnet("s1"))
        self.enc_aux = AuxEncoder(AUX_CH, self.AUX_DIMS)
        self.fuse = nn.ModuleList([FuseBlock(2 * rc + ac, ch) for rc, ac in zip(self.RES_CH, self.AUX_DIMS)])
        self.uper = UPerHead(ch, len(self.RES_CH))
        self.aux_head = nn.Conv2d(ch, num_classes, 1)          # deep supervision @ 1/4
        self.aux16_head = nn.Conv2d(ch, num_classes, 1)        # deep supervision @ 1/16
        self.up_half = nn.Sequential(cna(ch + 64 + 64, 128), cna(128, 128))
        self.full_spec = nn.Sequential(nn.Conv2d(FULL_CH, 64, 1, bias=False), nn.GroupNorm(8, 64), nn.GELU(),
                                       nn.Conv2d(64, 64, 1, bias=False), nn.GroupNorm(8, 64), nn.GELU())
        self.full_local = cna(FULL_CH, 32)
        self.up_full = nn.Sequential(cna(128 + 96, 64), cna(64, 64))
        self.drop = nn.Dropout2d(0.1)
        self.head = nn.Conv2d(64, num_classes, 1)
        self.coarse_head = nn.Conv2d(64, 4, 1)
        self.ordinal_head = nn.Conv2d(64, 2, 1)
        self.register_buffer("s2_mean", torch.tensor(S2_MEAN).view(1, -1, 1, 1))
        self.register_buffer("s2_std", torch.tensor(S2_STD).view(1, -1, 1, 1))

    def backbone_modules(self):
        return [self.enc_s2, self.enc_s1]

    def forward(self, b):
        s2, s1, aux = b["s2"], b["s1"], b["aux"]
        h2, f2 = self.enc_s2(s2)
        h1, f1 = self.enc_s1(s1)
        fa = self.enc_aux(aux)
        fused = [fb(a, c, d) for fb, a, c, d in zip(self.fuse, f2, f1, fa)]
        dec = self.uper(fused)
        x = F.interpolate(dec, scale_factor=2, mode="bilinear", align_corners=False)
        x = self.up_half(torch.cat([x, h2, h1], dim=1))
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        valid2 = aux[:, 7:8]
        full_in = torch.cat([((s2 - self.s2_mean) / self.s2_std) * valid2, s1, aux], dim=1)
        x = self.up_full(torch.cat([x, self.full_spec(full_in), self.full_local(full_in)], dim=1))
        x = self.drop(x)
        return {"logits": self.head(x), "coarse": self.coarse_head(x), "ordinal": self.ordinal_head(x),
                "aux": self.aux_head(dec), "aux16": self.aux16_head(fused[2])}


def freeze_backbone_bn(net):
    """Keep BatchNorm statistics of the pre-trained encoders fixed (call after net.train())."""
    for m in net.backbone_modules():
        for mod in m.modules():
            if isinstance(mod, nn.BatchNorm2d):
                mod.eval()


def set_backbone_trainable(net, flag: bool):
    for m in net.backbone_modules():
        for p in m.parameters():
            p.requires_grad_(flag)


def compose_probs(out):
    p_main = F.softmax(out["logits"].float(), dim=1)
    pc = F.softmax(out["coarse"].float(), dim=1)
    z = out["ordinal"].float()
    ge1 = torch.sigmoid(z[:, 0])
    ge2 = ge1 * torch.sigmoid(z[:, 1])
    p_h = torch.stack([pc[:, 0], pc[:, 1] * ge2, pc[:, 1] * (ge1 - ge2), pc[:, 1] * (1 - ge1),
                       pc[:, 2], pc[:, 3]], dim=1)
    return p_main, p_h



if not STAGE_A_CKPT.exists():
    raise FileNotFoundError(f"{STAGE_A_CKPT} not found. Run notebook 12 first (it writes ptuper_best.pt), "
                            f"or copy that file from the machine that ran it.")
model_A = PTUPerNet().to(device)
ckpt_A = torch.load(STAGE_A_CKPT, map_location=device, weights_only=False)
model_A.load_state_dict(ckpt_A["model"])
model_A.eval()
for p in model_A.parameters():
    p.requires_grad_(False)
print(f"Stage A loaded from {STAGE_A_CKPT} (epoch {ckpt_A.get('epoch', '?')}, "
      f"validation macro-F1 {100 * ckpt_A.get('val_macro_f1', float('nan')):.2f})")

STAGE_A_FEAT = 64
_cap = {}
model_A.up_full.register_forward_hook(lambda module, inputs, output: _cap.__setitem__("x", output))
SPEC_IN = STAGE_A_FEAT + AUX_CH + 2      # features + auxiliary stack (incl. terrain) + Sentinel-1
print(f"Specialist input channels: {SPEC_IN} = {STAGE_A_FEAT} (stage-A features) + {AUX_CH} (aux/terrain) + 2 (radar)")

## 5. Stage B: the Soil-Severity Specialist

- **Input:** stage A's 64-channel features, the 15 auxiliary channels (spectral indices, validity flags, terrain) and the 2 radar channels.
- **Full-resolution stem** keeps local detail. A **¼-resolution context branch** (two strided convolutions, four residual dilated blocks with dilations 1, 2, 4, 8, plus a global-pooling term) looks at a wide neighbourhood. The two are fused at full resolution.
- **Two heads on the same features:** an ordinal head (`P(≥ moderate)`, `P(serious | ≥ moderate)`) and a plain 3-way grade head. Their grade probabilities are averaged.
- **Regularisation against the stacking effect:** channel dropout on the input and Gaussian noise on stage A's features during training.
- **Grade order everywhere is (serious, moderate, slight)**, i.e. classes 1, 2, 3 of the 6-class map.

In [ ]:
class DilBlock(nn.Module):
    """Residual block with a dilated 3x3 convolution (large receptive field at low cost)."""
    def __init__(self, c: int, d: int):
        super().__init__()
        self.c1 = nn.Conv2d(c, c, 3, padding=d, dilation=d, bias=False)
        self.n1 = nn.GroupNorm(8, c)
        self.c2 = nn.Conv2d(c, c, 3, padding=1, bias=False)
        self.n2 = nn.GroupNorm(8, c)

    def forward(self, x):
        y = F.gelu(self.n1(self.c1(x)))
        y = self.n2(self.c2(y))
        return F.gelu(x + y)


class SoilSpecialist(nn.Module):
    def __init__(self, cin: int):
        super().__init__()
        self.in_drop = nn.Dropout2d(0.1)
        self.stem = nn.Sequential(cna(cin, 64), cna(64, 64))
        self.down = nn.Sequential(cna(64, 96, s=2), cna(96, 128, s=2))
        self.ctx = nn.Sequential(*[DilBlock(128, d) for d in (1, 2, 4, 8)])
        self.gp = nn.Sequential(nn.Linear(128, 128), nn.GELU())
        self.fuse = nn.Sequential(cna(64 + 128, 96), nn.Dropout2d(0.1), cna(96, 64))
        self.ord_head = nn.Conv2d(64, 2, 1)
        self.grade_head = nn.Conv2d(64, 3, 1)

    def forward(self, x):
        x = self.in_drop(x)
        s = self.stem(x)
        c = self.ctx(self.down(s))
        c = c + self.gp(c.mean((2, 3)))[:, :, None, None]
        c = F.interpolate(c, size=s.shape[-2:], mode="bilinear", align_corners=False)
        h = self.fuse(torch.cat([s, c], dim=1))
        return {"ord": self.ord_head(h), "grade": self.grade_head(h)}


def spec_probs(out):
    """Grade probabilities (B, 3, H, W) in the order (serious, moderate, slight)."""
    z = out["ord"].float()
    ge1 = torch.sigmoid(z[:, 0])                       # P(at least moderate)
    ge2 = ge1 * torch.sigmoid(z[:, 1])                 # P(serious)
    p_ord = torch.stack([ge2, ge1 - ge2, 1 - ge1], dim=1)
    p_cls = F.softmax(out["grade"].float(), dim=1)
    return 0.5 * (p_ord + p_cls)


def build_input(feat, b, drop_terrain=False, noise=0.0):
    f = feat.float()
    if noise > 0:
        f = f + noise * f.std() * torch.randn_like(f)
    x = torch.cat([f, b["aux"], b["s1"]], dim=1)
    if drop_terrain:
        x[:, STAGE_A_FEAT + AUX_CH - N_DEM:STAGE_A_FEAT + AUX_CH] = 0
    return x


spec = SoilSpecialist(SPEC_IN).to(device)
print(f"SoilSpecialist: {sum(p.numel() for p in spec.parameters()) / 1e6:.2f}M parameters")

# --- smoke test ----------------------------------------------------------------------------------
_b = {"s2": torch.rand(2, 11, PATCH, PATCH, device=device) * 0.3,
      "s1": torch.randn(2, 2, PATCH, PATCH, device=device),
      "aux": torch.randn(2, AUX_CH, PATCH, PATCH, device=device)}
if device.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
with torch.no_grad(), autocast():
    _outA = model_A(_b)
    _feat = _cap["x"]
spec.train()
with autocast():
    _outS = spec(build_input(_feat, _b))
    _loss = sum(v.float().mean() for v in _outS.values())
_loss.backward()
spec.zero_grad(set_to_none=True)
print("Stage-A feature map:", tuple(_feat.shape), "| specialist outputs:", {k: tuple(v.shape) for k, v in _outS.items()})
if device.type == "cuda":
    peak = torch.cuda.max_memory_allocated() / 2**30
    print(f"Peak GPU memory, batch of 2 @ {PATCH}px: {peak:.2f} GB (batch {BATCH_SIZE} is roughly {peak * BATCH_SIZE / 2:.1f} GB)")
del _b, _outA, _outS, _feat, _loss
if device.type == "cuda":
    torch.cuda.empty_cache()

## 6. Loss and metrics

**Loss, on soil pixels only** (pixels whose ground truth is serious, moderate or slight):

`weighted CE with ordinal label smoothing  +  BCE(P(≥ moderate))  +  BCE(P(serious | ≥ moderate))`

Class weights are the inverse square root of the grade frequencies in the training scenes. The BCE terms use the positive-class weights from the dataset cell. A batch with no soil pixel contributes zero loss.

**Metrics.** The 6-class confusion, precision/recall/F1 and report functions are the same as in notebooks 10 to 14. For grading, a 3×3 confusion over true-soil pixels measures how well each model picks the grade given that the pixel really is soil.

In [ ]:
def confusion(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    keep = (y_true != IGNORE) & (y_pred != IGNORE)
    return np.bincount(NUM_CLASSES * y_true[keep].astype(np.int64) + y_pred[keep].astype(np.int64),
                       minlength=NUM_CLASSES ** 2).reshape(NUM_CLASSES, NUM_CLASSES)


def prf(cm: np.ndarray):
    tp = np.diag(cm).astype(np.float64)
    pr = tp / np.maximum(cm.sum(0), 1)
    rc = tp / np.maximum(cm.sum(1), 1)
    f1 = 2 * pr * rc / np.maximum(pr + rc, 1e-12)
    return pr, rc, f1


def macro_f1(cm: np.ndarray, classes=range(NUM_CLASSES)) -> float:
    f1 = prf(cm)[2]
    present = [k for k in classes if cm[k].sum() > 0]
    return float(np.mean([f1[k] for k in present])) if present else 0.0


def report(cm: np.ndarray, title: str) -> dict:
    pr, rc, f1 = prf(cm)
    support = cm.sum(1)
    line = "=" * 86
    print(f"\n{line}\n{title}\n{line}")
    print(f"{'Cls':<4}{'Name':<36}{'Pr (%)':>9}{'Rc (%)':>9}{'F1 (%)':>9}{'Pixels':>16}")
    print("-" * 86)
    for k in range(NUM_CLASSES):
        print(f"{k:<4}{CLASS_NAMES[k]:<36}{100 * pr[k]:>9.2f}{100 * rc[k]:>9.2f}{100 * f1[k]:>9.2f}{support[k]:>16,}")
    print("-" * 86)
    w5 = support[:5] / max(support[:5].sum(), 1)
    out = {
        "precision": pr.tolist(), "recall": rc.tolist(), "f1": f1.tolist(),
        "macro_f1_all": macro_f1(cm), "macro_f1_water": macro_f1(cm, range(5)),
        "weighted_water": [float((pr[:5] * w5).sum()), float((rc[:5] * w5).sum()), float((f1[:5] * w5).sum())],
    }
    wp, wr, wf = out["weighted_water"]
    print(f"{'':4}{'MACRO-F1, all 6 classes':<36}{'':>18}{100 * out['macro_f1_all']:>9.2f}")
    print(f"{'':4}{'MACRO-F1, water-related (raw 2-6)':<36}{'':>18}{100 * out['macro_f1_water']:>9.2f}")
    print(f"{'':4}{'Pixel-weighted, raw 2-6 (Table 1)':<36}{100 * wp:>9.2f}{100 * wr:>9.2f}{100 * wf:>9.2f}")
    print(line)
    return out



def conf3(t: np.ndarray, p: np.ndarray) -> np.ndarray:
    return np.bincount(3 * t.astype(np.int64) + p.astype(np.int64), minlength=9).reshape(3, 3)


def grade_macro_f1(cm: np.ndarray) -> float:
    f1 = prf(cm)[2]
    present = [k for k in range(3) if cm[k].sum() > 0]
    return float(np.mean([f1[k] for k in present])) if present else 0.0


GRADE_NAMES = ["Soil seriously affected", "Soil moderately affected", "Soil slightly affected"]


class SoilLoss(nn.Module):
    def __init__(self, grade_w, ord_pos_w, eps=ORDINAL_SMOOTH):
        super().__init__()
        S = torch.tensor([[1 - eps, eps, 0.0], [eps / 2, 1 - eps, eps / 2], [0.0, eps, 1 - eps]])
        self.register_buffer("S", S)                       # soft targets, rows = serious / moderate / slight
        self.register_buffer("w", grade_w)
        self.register_buffer("opw", ord_pos_w)

    def forward(self, out, y):
        soil = (y >= 1) & (y <= 3)
        if not soil.any():
            return out["ord"].float().sum() * 0.0, {}
        g = (y - 1).clamp(0, 2)                           # 0 serious, 1 moderate, 2 slight
        rank = 2 - g                                      # slight 0, moderate 1, serious 2
        logits = out["grade"].float().permute(0, 2, 3, 1)[soil]
        logp = F.log_softmax(logits, dim=1)
        gt = g[soil]
        w = self.w[gt]
        ce = (-(self.S[gt] * logp).sum(1) * w).sum() / w.sum()
        z = out["ord"].float()
        b1 = F.binary_cross_entropy_with_logits(z[:, 0][soil], (rank[soil] >= 1).float(), pos_weight=self.opw[0])
        m2 = soil & (rank >= 1)
        b2 = z.sum() * 0.0
        if m2.any():
            b2 = F.binary_cross_entropy_with_logits(z[:, 1][m2], (rank[m2] >= 2).float(), pos_weight=self.opw[1])
        return ce + b1 + b2, {"ce": ce.item(), "ord1": b1.item(), "ord2": float(b2)}


grade_counts = np.array([train_pixels[1], train_pixels[2], train_pixels[3]], dtype=np.float64)
grade_w = (grade_counts.sum() / grade_counts) ** 0.5
grade_w = torch.tensor(grade_w / grade_w.mean(), dtype=torch.float32)
criterion = SoilLoss(grade_w, ORD_POS_WEIGHT).to(device)
print("Soil grade weights (serious, moderate, slight):", np.round(grade_w.numpy(), 2).tolist())


@torch.no_grad()
def eval_grading(loader, drop_terrain=False):
    """Validation: grade-only quality on true-soil pixels, specialist vs stage A's own grade choice."""
    spec.eval()
    cm_a, cm_s = np.zeros((3, 3), dtype=np.int64), np.zeros((3, 3), dtype=np.int64)
    total, n = 0.0, 0
    for b in loader:
        b = to_dev(b)
        with autocast():
            out_a = model_A(b)
            out_s = spec(build_input(_cap["x"], b, drop_terrain=drop_terrain))
        loss, _ = criterion(out_s, b["y"])
        total += loss.item(); n += 1
        p_a, _ = compose_probs(out_a)
        p_s = spec_probs(out_s)
        y = b["y"].cpu().numpy()
        soil = (y >= 1) & (y <= 3)
        t = (y - 1)[soil]
        cm_a += conf3(t, p_a[:, 1:4].argmax(1).cpu().numpy()[soil])
        cm_s += conf3(t, p_s.argmax(1).cpu().numpy()[soil])
    return total / max(n, 1), cm_a, cm_s

## 7. Training the specialist

Stage A runs under `torch.no_grad()` for every batch to produce the features; only the specialist's weights are updated (AdamW, one-cycle learning rate, gradient clipping, mixed precision). Each epoch is validated on fixed soil-rich patches of the two held-out training scenes, which stage A never saw. The log compares, **on the same true-soil pixels**, the grade macro-F1 of stage A's own choice (`A`) and of the specialist (`S`). The best epoch is the one with the highest specialist grade macro-F1.

In [ ]:
steps_per_epoch = math.ceil(SAMPLES_PER_EPOCH / BATCH_SIZE)
optimizer = torch.optim.AdamW(spec.parameters(), lr=SPEC_LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=SPEC_LR, total_steps=EPOCHS * steps_per_epoch,
                                                pct_start=WARMUP_FRAC)
scaler = make_scaler()

run_cfg = {"epochs": EPOCHS, "samples": SAMPLES_PER_EPOCH, "batch": BATCH_SIZE, "patch": PATCH, "lr": SPEC_LR,
           "val_scenes": VAL_SCENES, "stage_a_epoch": ckpt_A.get("epoch"), "dem": USE_DEM}
history = {"train_loss": [], "val_loss": [], "grade_f1_spec": [], "grade_f1_A": [], "grade_f1_per_class": []}
start_epoch, best_score = 1, -1.0

if RESUME and SPEC_LAST.exists():
    st = torch.load(SPEC_LAST, map_location=device, weights_only=False)
    if st.get("run_cfg") == run_cfg:
        spec.load_state_dict(st["model"]); optimizer.load_state_dict(st["optimizer"])
        scheduler.load_state_dict(st["scheduler"]); scaler.load_state_dict(st["scaler"])
        history, best_score, start_epoch = st["history"], st["best_score"], st["epoch"] + 1
        print(f"Resuming from epoch {start_epoch} (best specialist grade macro-F1 so far {100 * best_score:.2f})")
    else:
        print("Found a checkpoint from a different configuration - starting fresh.")

print(f"Training the specialist: epochs {start_epoch}..{EPOCHS} | {steps_per_epoch} steps/epoch | batch {BATCH_SIZE} | patch {PATCH}")

for epoch in range(start_epoch, EPOCHS + 1):
    t0 = time.time()
    picks = train_index.sample(SAMPLES_PER_EPOCH, np.random.default_rng(SEED + 1000 * epoch))
    loader = make_loader(PatchDataset(train_scenes, train_index, picks, PATCH, train=True,
                                      jitter=GRID_STRIDE // 2, seed=SEED + epoch))
    spec.train()
    running, parts_sum = 0.0, {}
    for step, b in enumerate(loader, 1):
        b = to_dev(b)
        with torch.no_grad(), autocast():
            model_A(b)                                    # fills _cap["x"] with stage A's features
        with autocast():
            out = spec(build_input(_cap["x"], b, noise=FEAT_NOISE))
        loss, parts = criterion(out, b["y"])
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(spec.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        running += loss.item()
        for k, v in parts.items():
            parts_sum[k] = parts_sum.get(k, 0.0) + v
        if step % 25 == 0:
            print(f"  epoch {epoch} step {step}/{len(loader)} loss {running / step:.4f} "
                  f"({(time.time() - t0) / step:.2f}s/step)", end="\r")

    train_loss = running / len(loader)
    val_loss, cm_a, cm_s = eval_grading(val_loader)
    f_a, f_s = grade_macro_f1(cm_a), grade_macro_f1(cm_s)
    history["train_loss"].append(train_loss); history["val_loss"].append(val_loss)
    history["grade_f1_spec"].append(f_s); history["grade_f1_A"].append(f_a)
    history["grade_f1_per_class"].append([float(v) for v in prf(cm_s)[2]])

    improved = f_s > best_score
    if improved:
        best_score = f_s
        torch.save({"model": spec.state_dict(), "epoch": epoch, "grade_f1": f_s}, SPEC_BEST)
    torch.save({"model": spec.state_dict(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict(), "epoch": epoch, "history": history, "best_score": best_score,
                "run_cfg": run_cfg}, SPEC_LAST)
    parts_txt = " ".join(f"{k} {v / len(loader):.3f}" for k, v in parts_sum.items())
    print(f"Epoch {epoch:2d}/{EPOCHS} | train {train_loss:.4f} ({parts_txt}) | val {val_loss:.4f} | "
          f"grade macro-F1  A {100 * f_a:5.2f}  S {100 * f_s:5.2f}  ({100 * (f_s - f_a):+.2f}) | "
          f"S per grade " + " ".join(f"{100 * v:4.1f}" for v in prf(cm_s)[2])
          + f" | {(time.time() - t0) / 60:.1f} min{' *' if improved else ''}")

print(f"\nTraining complete. Best specialist grade macro-F1 on held-out soil pixels: {100 * best_score:.2f} -> {SPEC_BEST}")

## 8. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
ep = np.arange(1, len(history["train_loss"]) + 1)
axes[0].plot(ep, history["train_loss"], label="train")
axes[0].plot(ep, history["val_loss"], label="val", linestyle="--")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].set_title("Specialist loss (soil pixels)")
axes[0].legend(); axes[0].grid(True)
axes[1].plot(ep, 100 * np.array(history["grade_f1_spec"]), label="specialist (S)", linewidth=2)
axes[1].plot(ep, 100 * np.array(history["grade_f1_A"]), label="stage A's own grade choice (A)", linestyle="--")
per = np.array(history["grade_f1_per_class"])
for k in range(3):
    axes[1].plot(ep, 100 * per[:, k], alpha=0.5, label=f"S: {GRADE_NAMES[k]}")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Grade F1 (%)")
axes[1].set_title("Grading quality on held-out true-soil pixels"); axes[1].legend(fontsize=8); axes[1].grid(True)
plt.tight_layout()
plt.show()

## 9. Ablation: do the terrain inputs matter for grading?

The best specialist is loaded and evaluated on the held-out validation patches with and without the terrain channels (inference-time ablation: the network never saw zeros there, so read the drop as an upper bound).

In [ ]:
spec.load_state_dict(torch.load(SPEC_BEST, map_location=device, weights_only=False)["model"])
spec.eval()

short = ["Water", "Soil serious", "Soil moderate", "Soil slight", "Vegetation", "Not affected"]
ablation = {}
for label, kw in [("full specialist", {}), ("terrain zeroed", {"drop_terrain": True})]:
    if label == "terrain zeroed" and not USE_DEM:
        continue
    _, cm_a, cm_s = eval_grading(val_loader, **kw)
    ablation[label] = {"grade_macro_f1": grade_macro_f1(cm_s), "per_grade_f1": prf(cm_s)[2].tolist()}
ablation["stage A's own grade choice"] = {"grade_macro_f1": grade_macro_f1(cm_a), "per_grade_f1": prf(cm_a)[2].tolist()}

print(f"{'Setting':<30}{'grade macro-F1':>16}  " + "".join(f"{g.replace('Soil ', ''):>22}" for g in GRADE_NAMES))
for label, r in ablation.items():
    print(f"{label:<30}{100 * r['grade_macro_f1']:>16.2f}  " + "".join(f"{100 * v:>22.2f}" for v in r["per_grade_f1"]))

## 10. Full-scene inference, choosing `beta`, and the final map

- Each scene is tiled into 512×512 centres predicted from 640×640 windows (64 px margin, reflect-padded), exactly like notebook 12. For every window, stage A and the specialist run together (flip test-time augmentation: both are averaged over the flipped passes after un-flipping).
- Per pixel the **fusion** is: `soil = pA[1] + pA[2] + pA[3]`, `grade = (1 − beta)·pA[1:4] / soil + beta·pS`, and the final probabilities are `pA` with `pA[1:4]` replaced by `soil · grade`. Water, vegetation and not-affected are untouched, so **only grading changes**.
- **`beta` is chosen on the two held-out training scenes** from `{0, 0.25, 0.5, 0.75, 1}` by 6-class macro-F1 on a 10 % pixel sample. If the best `beta` improves on `beta = 0` by less than `MIN_GAIN` (0.2 points), `beta = 0` is used and the result equals notebook 12's.
- No test data is used for any choice.

In [ ]:
def tile_starts(length: int, tile: int):
    starts = list(range(0, max(length - tile, 0) + 1, tile))
    if starts[-1] + tile < length:
        starts.append(length - tile)
    return starts


TTA_OPS = {"none": [(0, False)], "flip": [(0, False), (0, True)],
           "d4": [(k, f) for f in (False, True) for k in range(4)]}[TTA]


def _apply(x, k, flip):
    if flip:
        x = x.flip(-1)
    return torch.rot90(x, k, dims=(-2, -1)) if k else x


def _invert(x, k, flip):
    if k:
        x = torch.rot90(x, -k, dims=(-2, -1))
    return x.flip(-1) if flip else x


@torch.no_grad()
def predict_window(b):
    """-> stage-A flat probabilities (1,6,H,W) and specialist grade probabilities (1,3,H,W), TTA-averaged."""
    p_a = p_s = 0.0
    for k, flip in TTA_OPS:
        bb = {key: _apply(v, k, flip) for key, v in b.items()}
        with autocast():
            out_a = model_A(bb)
            out_s = spec(build_input(_cap["x"], bb))
        pa, _ = compose_probs(out_a)
        p_a = p_a + _invert(pa, k, flip)
        p_s = p_s + _invert(spec_probs(out_s), k, flip)
    return p_a / len(TTA_OPS), p_s / len(TTA_OPS)


@torch.no_grad()
def scene_outputs(sc: Scene, max_tiles=None):
    """Yield (r0, c0, pA (6,TILE,TILE), pS (3,TILE,TILE)) covering the whole scene."""
    spec.eval()
    model_A.eval()
    done = 0
    for r0 in tile_starts(sc.H, TILE):
        for c0 in tile_starts(sc.W, TILE):
            ra, rb = max(r0 - MARGIN, 0), min(r0 + TILE + MARGIN, sc.H)
            ca, cb = max(c0 - MARGIN, 0), min(c0 + TILE + MARGIN, sc.W)
            s2, s1, aux = make_inputs(*sc.read(ra, ca, rb - ra, cb - ca))
            pad = ((0, 0), (MARGIN - (r0 - ra), MARGIN - (rb - r0 - TILE)),
                   (MARGIN - (c0 - ca), MARGIN - (cb - c0 - TILE)))
            b = {k: torch.from_numpy(np.pad(v, pad, mode="reflect"))[None].to(device)
                 for k, v in (("s2", s2), ("s1", s1), ("aux", aux))}
            pa, ps = predict_window(b)
            crop = (slice(None), slice(MARGIN, MARGIN + TILE), slice(MARGIN, MARGIN + TILE))
            yield r0, c0, pa[0][crop].cpu().numpy(), ps[0][crop].cpu().numpy()
            done += 1
            if max_tiles and done >= max_tiles:
                return


def fuse(pa: np.ndarray, ps: np.ndarray, beta: float) -> np.ndarray:
    """Redistribute the soil probability mass using the specialist's grades. Class axis first."""
    if beta == 0.0:
        return pa
    soil = pa[1:4].sum(0, keepdims=True)
    within = pa[1:4] / (soil + 1e-8)
    out = pa.copy()
    out[1:4] = soil * ((1 - beta) * within + beta * ps)
    return out


rng = np.random.default_rng(SEED)
cal_a, cal_s, cal_y = [], [], []
for sc in val_scenes:
    t0 = time.time()
    for r0, c0, pa, ps in scene_outputs(sc, MAX_TILES):
        y = sc.gt[r0:r0 + TILE, c0:c0 + TILE]
        take = (y != IGNORE) & (rng.random(y.shape) < 0.10)
        cal_a.append(pa[:, take]); cal_s.append(ps[:, take]); cal_y.append(y[take])
    print(f"  {sc.name}: inference {(time.time() - t0) / 60:.1f} min")
cal_a = np.concatenate(cal_a, axis=1).astype(np.float32)
cal_s = np.concatenate(cal_s, axis=1).astype(np.float32)
cal_y = np.concatenate(cal_y)
print(f"Calibration sample: {len(cal_y):,} pixels ({int(((cal_y >= 1) & (cal_y <= 3)).sum()):,} soil)")

beta_scores = {b: macro_f1(confusion(cal_y, fuse(cal_a, cal_s, b).argmax(0))) for b in (0.0, 0.25, 0.5, 0.75, 1.0)}
best_beta = max(beta_scores, key=lambda b: (beta_scores[b], -b))
BETA = best_beta if beta_scores[best_beta] - beta_scores[0.0] >= MIN_GAIN else 0.0
print("Held-out macro-F1 by beta (0 = stage A only):", {b: round(100 * s, 2) for b, s in beta_scores.items()})
print(f"-> beta = {BETA}" + ("  (the specialist improved held-out macro-F1 enough to be used)" if BETA > 0 else
      f"  (gain below {100 * MIN_GAIN:.1f} points: the final map equals notebook 12's)"))
CAL_PATH.write_text(json.dumps({"beta": BETA, "beta_scores": {str(b): s for b, s in beta_scores.items()},
                                "min_gain": MIN_GAIN, "stage_a_checkpoint": str(STAGE_A_CKPT)}, indent=2))
del cal_a, cal_s, cal_y

## 11. Evaluation on the 9 test scenes

- **Stage A only** (`beta = 0`) should reproduce notebook 12's flat-head result (macro-F1 78.62, pixel-weighted F1 84.12), which is a useful sanity check that the checkpoint and pipeline are correct.
- **Final** uses the `beta` chosen on the held-out scenes.
- **Grading-only comparison:** among pixels that are truly soil, how often each model picks the right grade (stage A's own, the specialist's alone, and the mix).

In [ ]:
test_scenes = all_test[:2] if QUICK_TEST else all_test

cm_A = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
cm_final = np.zeros_like(cm_A)
cm_gA, cm_gS, cm_gMix = (np.zeros((3, 3), dtype=np.int64) for _ in range(3))
pred_maps, per_scene = {}, {}
for sc in test_scenes:
    t0 = time.time()
    p_A = np.full((sc.H, sc.W), IGNORE, dtype=np.uint8)
    p_F = np.full((sc.H, sc.W), IGNORE, dtype=np.uint8)
    for r0, c0, pa, ps in scene_outputs(sc, MAX_TILES):
        sl = (slice(r0, r0 + TILE), slice(c0, c0 + TILE))
        p_A[sl] = pa.argmax(0)
        p_F[sl] = fuse(pa, ps, BETA).argmax(0)
        y = sc.gt[sl]
        soil = (y >= 1) & (y <= 3)
        if soil.any():
            t = (y - 1)[soil]
            cm_gA += conf3(t, pa[1:4].argmax(0)[soil])
            cm_gS += conf3(t, ps.argmax(0)[soil])
            cm_gMix += conf3(t, ((1 - BETA) * pa[1:4] / (pa[1:4].sum(0, keepdims=True) + 1e-8) + BETA * ps).argmax(0)[soil])
    cm_A += confusion(sc.gt, p_A)
    cm_scene = confusion(sc.gt, p_F)
    cm_final += cm_scene
    pred_maps[sc.name] = p_F
    np.save(PRED_DIR / f"{sc.name}.npy", np.stack([p_A, p_F]))
    per_scene[sc.name] = {"macro_f1": macro_f1(cm_scene), "f1": prf(cm_scene)[2].tolist(),
                          "stage_a_macro_f1": macro_f1(confusion(sc.gt, p_A))}
    print(f"  {sc.name}: {(time.time() - t0) / 60:.1f} min | scene macro-F1 stage A {100 * per_scene[sc.name]['stage_a_macro_f1']:.2f}"
          f" -> final {100 * macro_f1(cm_scene):.2f}")

res_A = report(cm_A, "TEST SET - stage A only (notebook 12's model, flat head, raw argmax)")
res_final = report(cm_final, f"TEST SET - stage A + Soil-Severity Specialist (beta = {BETA})")

print("\nGrading-only comparison on TRUE soil pixels (macro-F1 over serious / moderate / slight)")
print(f"{'Method':<34}{'macro-F1':>10}  " + "".join(f"{g.replace('Soil ', ''):>22}" for g in GRADE_NAMES))
grading = {}
for label, cm in (("Stage A's own grade choice", cm_gA), ("Specialist alone", cm_gS), (f"Mix (beta = {BETA})", cm_gMix)):
    grading[label] = {"macro_f1": grade_macro_f1(cm), "per_grade_f1": prf(cm)[2].tolist()}
    print(f"{label:<34}{100 * grading[label]['macro_f1']:>10.2f}  " + "".join(f"{100 * v:>22.2f}" for v in grading[label]["per_grade_f1"]))

columns = {   # real test results of the earlier notebooks (per-class F1, macro-F1, pixel-weighted F1 raw 2-6)
    "NB10 cal": ([98.62, 71.04, 60.20, 63.54, 67.22, 98.56], 76.53, 81.53),
    "NB12 raw": ([98.50, 69.63, 62.34, 66.72, 75.58, 98.99], 78.62, 84.12),
}
columns["NB15 stageA"] = ([100 * v for v in res_A["f1"]], 100 * res_A["macro_f1_all"], 100 * res_A["weighted_water"][2])
columns["NB15 final"] = ([100 * v for v in res_final["f1"]], 100 * res_final["macro_f1_all"], 100 * res_final["weighted_water"][2])
print("\nPer-class F1 (%) on the test set")
print(f"{'Class':<36}" + "".join(f"{c:>13}" for c in columns))
for k in range(NUM_CLASSES):
    print(f"{CLASS_NAMES[k]:<36}" + "".join(f"{v[0][k]:>13.2f}" for v in columns.values()))
print(f"{'MACRO-F1 (6 classes)':<36}" + "".join(f"{v[1]:>13.2f}" for v in columns.values()))
print(f"{'Pixel-weighted F1, raw 2-6':<36}" + "".join(f"{v[2]:>13.2f}" for v in columns.values()))
print("Paper Table 1 best (RUPNet), pixel-weighted F1 raw 2-6: 81.19")

RESULTS_PATH.write_text(json.dumps({
    "model": "Soil-Severity Specialist on frozen notebook-12 model (notebook 15)", "quick_test": QUICK_TEST,
    "beta": BETA, "beta_scores": {str(b): s for b, s in beta_scores.items()}, "test_stage_a": res_A,
    "test_final": res_final, "grading_only": grading, "per_scene": per_scene, "val_ablation": ablation,
    "best_val_grade_macro_f1": best_score, "confusion_final": cm_final.tolist(),
    "config": {**run_cfg, "tta": TTA, "tile": TILE, "margin": MARGIN},
}, indent=2))
print(f"\nSaved all numbers -> {RESULTS_PATH}")

## 12. Confusion matrix

In [ ]:
cm_pct = 100 * cm_final / np.maximum(cm_final.sum(1, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(cm_pct, cmap="Blues", vmin=0, vmax=100)
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(short, rotation=35, ha="right")
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(short)
ax.set_xlabel("Predicted"); ax.set_ylabel("Ground truth")
ax.set_title("Test confusion matrix, stage A + Soil-Severity Specialist (% of each true class)")
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, f"{cm_pct[i, j]:.1f}", ha="center", va="center",
                color="white" if cm_pct[i, j] > 55 else "black", fontsize=9)
fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

## 13. Prediction maps

For up to three test scenes, the 512×512 window richest in affected soil is shown four ways: true colour, the 1.5 km terrain position feature (blue = depression), the ground truth and the final prediction.

In [ ]:
PALETTE = ["#1f77b4", "#d62728", "#ff7f0e", "#f2d024", "#2ca02c", "#e6e6e6", "#7f7f7f"]
cmap = ListedColormap(PALETTE)


def to_display(label_map: np.ndarray) -> np.ndarray:
    return np.where(label_map == IGNORE, NUM_CLASSES, label_map)


def stretch_rgb(rgb: np.ndarray) -> np.ndarray:
    out = np.empty_like(rgb, dtype=np.float32)
    for k in range(3):
        lo, hi = np.percentile(rgb[..., k], [2, 98])
        out[..., k] = np.clip((rgb[..., k] - lo) / (hi - lo + 1e-8), 0, 1)
    return out


shown = [sc for sc in test_scenes if sc.name in pred_maps][:3]
fig, axes = plt.subplots(len(shown), 4, figsize=(20, 5 * len(shown)), squeeze=False)
for row, sc in enumerate(shown):
    pred = pred_maps[sc.name]
    block = 256
    soil = ((sc.gt >= 1) & (sc.gt <= 3)) & (pred != IGNORE)
    hb, wb = sc.H // block, sc.W // block
    counts = soil[:hb * block, :wb * block].reshape(hb, block, wb, block).sum((1, 3))
    bi, bj = np.unravel_index(counts.argmax(), counts.shape)
    r0 = int(np.clip(bi * block - 128, 0, sc.H - 512))
    c0 = int(np.clip(bj * block - 128, 0, sc.W - 512))
    s2, _, dem = sc.read(r0, c0, 512, 512)
    axes[row, 0].imshow(stretch_rgb(np.stack([s2[3], s2[2], s2[1]], -1).astype(np.float32)))
    axes[row, 0].set_title(f"{sc.name} - S2 true colour")
    if dem is not None:
        axes[row, 1].imshow(dem[4].astype(np.float32), cmap="RdBu", vmin=-3, vmax=3)
        axes[row, 1].set_title("Terrain: TPI 1.5 km (blue = depression)")
    else:
        axes[row, 1].set_title("no terrain (USE_DEM = False)")
    axes[row, 2].imshow(to_display(sc.gt[r0:r0 + 512, c0:c0 + 512]), cmap=cmap, vmin=0, vmax=NUM_CLASSES,
                        interpolation="nearest")
    axes[row, 2].set_title("Ground truth")
    axes[row, 3].imshow(to_display(pred[r0:r0 + 512, c0:c0 + 512]), cmap=cmap, vmin=0, vmax=NUM_CLASSES,
                        interpolation="nearest")
    axes[row, 3].set_title("Stage A + specialist prediction")
    for a in axes[row]:
        a.axis("off")
handles = [plt.Rectangle((0, 0), 1, 1, color=PALETTE[k]) for k in range(NUM_CLASSES + 1)]
fig.legend(handles, short + ["Cloud / shadow / not predicted"], loc="lower center", ncol=7, fontsize=9, frameon=False)
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.show()

## 14. Summary: fill in after running

| Test set | NB10 (cal) | NB12 (raw) | NB15 stage A | NB15 final |
|---|---|---|---|---|
| Water F1 | 98.62 | 98.50 | | |
| Soil seriously affected F1 | 71.04 | 69.63 | | |
| Soil moderately affected F1 | 60.20 | 62.34 | | |
| Soil slightly affected F1 | 63.54 | 66.72 | | |
| Vegetation affected F1 | 67.22 | 75.58 | | |
| Not affected F1 | 98.56 | 98.99 | | |
| Macro-F1 (6 classes) | 76.53 | 78.62 | | |
| Pixel-weighted F1, raw 2–6 (paper: 81.19) | 81.53 | 84.12 | | |

**Grading-only (true soil pixels, macro-F1 over the three grades):** stage A ___ / specialist ___ / mix ___

**`beta` chosen on held-out scenes:** ___ (0 means the specialist was not used)

**How to read the result.**
- If *stage A only* does not reproduce notebook 12's numbers, check that `ptuper_best.pt` is the checkpoint from the full run (not a quick test) and that the same `configs/data.yaml` and terrain files are used.
- If the specialist **grades better on true-soil pixels but the final score barely moves**, the limit is soil *detection* (stage A misclassifying soil as vegetation or background), not grading. The next lever is then the detection side.
- If **neither moves**, the serious/moderate/slight boundary is limited by label consistency, which is a legitimate finding for the report.

**Settings to report:** stage-A checkpoint epoch, `beta`, TTA mode, GPU and wall-clock time (all in `outputs/soil_results.json`).